### 5. Qual é a taxa de conversão de Recuperações Judiciais em Falência (convolação em falência por descumprimento do plano ou rejeição da assembleia)?

**Pergunta:** qual a fatia de empresas que ajuizam RJ e têm a falência decretada nos mesmos autos?

**Fonte no mart (consumo puro):** `gold_base_analitica` — a convolação é
identificada pela presença de `'data_decretao_falencia'` nos autos da RJ
(movimento TPU 202 "Decretação de falência" registrado dentro do processo
de classe 129). O tempo até a convolação já vem calculado em
`'dias_ate_convolucao'` (ajuizamento → decretação) e o denominador
condicionado usa `'data_processamento_efetivo'` (a mesma concessão validada
na pergunta 1). Este notebook não refaz cálculos: consome o resultado pronto.

**Recorte:** classe 129 (pedido principal de RJ), ajuizado a partir de
09/06/2005. Dois denominadores explícitos:
- **Taxa bruta** — sobre todas as RJs do corpus;
- **Taxa condicionada** — só sobre as RJs com concessão identificável
  (recomendada: a convolação por descumprimento do plano pressupõe que o
  processo chegou à fase de processamento).

**Sobre as causas:** a pergunta menciona duas situações que levam à falência — a rejeição do plano pelos credores na assembleia e o descumprimento do plano depois de aprovado. O Datajud não registra **por que** a falência foi decretada: a justificativa fica dentro do texto da decisão, e não existe um movimento próprio na TPU para cada
causa (a mesma limitação estrutural da pergunta 2). O que medimos, com segurança, é a **taxa total** de convolação. A causa pode ser apenas **aproximada pelo tempo**: quando a falência é decretada sem que o processamento tenha sido concedido, o mais provável é rejeição ou indeferimento (art. 73, V); quando a decretação vem anos depois da concessão, o perfil é de descumprimento do plano durante sua execução. Essa separação é uma inferência temporal, não uma medida direta — e por
isso está declarada como limitação.

**Limitações:**
1. A causa da convolação (descumprimento vs rejeição) é inferência temporal,
   não medida direta;
2. Convolações em autos de falência separados (classe 108 aberta depois de
   indeferir a RJ) não têm vínculo pai-filho no Datajud — ficam fora da
   taxa, que é, portanto, um piso;
3. O denominador condicionado herda a cobertura do marco de concessão
   (42,4% — pergunta 1).



#### O que esta consulta mede e como 
**Nota terminológica:** o título emprega "conversão" no sentido corrente
e "convolação" no sentido jurídico consagrado pela doutrina falimentar
(Lei 11.101/2005, art. 73). O texto deste notebook usa convolação. As
colunas do mart preservam a grafia `convolucao` (artefato da camada
gold, congelada) e são citadas apenas entre crases.

**Entrada:** todos os campos vêm da tabela `default.gold_base_analitica`,
já filtrada para os processos de Recuperação Judicial (classe 129)
ajuizados no período da lei vigente (a partir de 09/06/2005).

A consulta lê quatro colunas por processo:

| Coluna lida | O que representa |
|---|---|
| `numero_processo` | identificador do processo (usado apenas para a contagem) |
| `data_decretao_falencia` | a data da decretação de falência registrada nos próprios autos da RJ (nulo = a RJ não foi convolada) |
| `dias_ate_convolucao` | dias entre o ajuizamento e a decretação, já calculado na gold |
| `data_processamento_efetivo` | a data em que o juiz concedeu o processamento da RJ (nulo = concessão não identificável) |

**Saída — o que ela produz (uma única linha com oito números):**

| Coluna produzida | Como é calculada | O que significa |
|---|---|---|
| `total_rj` | conta todos os processos do recorte | o denominador da taxa bruta (5.709) |
| `com_concessao` | conta os processos em que a concessão está datada | o denominador da taxa condicionada (2.422) |
| `convoladas` | conta os casos em que a decretação existe nos autos | o numerador das duas taxas (177) |
| `taxa_bruta_pct` | convoladas ÷ total_rj × 100 | a taxa bruta de convolação (3,10%) |
| `taxa_condicionada_pct` | convoladas com concessão ÷ com_concessao × 100 | a taxa condicionada, métrica recomendada (5,12%) |
| `mediana_dias_ate_convolacao` | mediana de `dias_ate_convolucao`, só para as convoladas | o tempo típico até a decretação (1.623 dias) — medido sobre o conjunto completo das 177, incluindo as 53 sem concessão |
| `media_dias_ate_convolacao` | média de `dias_ate_convolucao`, só para as convoladas | o tempo médio até a decretação (1.980 dias), mais sensível aos casos extremos |
| `pct_convolacao_antes_da_concessao` | convoladas sem concessão ÷ convoladas × 100 | a fatia candidata a rejeição/indeferimento (art. 73, V) (29,9% — 53 de 177) |

**Em uma frase:** a consulta pega as 5.709 RJs do período da lei vigente,
verifica em quantas houve decretação de falência nos próprios autos e em
quantas dessas o processamento havia sido concedido, produzindo os oito
números que sustentam a resposta: a taxa bruta, a taxa
condicionada, o tempo típico até a decretação
(mediana) e a fatia de convolações ocorridas antes de
qualquer concessão registrada.

In [0]:
%sql
-- Q5 · TAXA DE CONVOLACAO: RJ -> falencia nos mesmos autos (nacional)
WITH base AS (
  SELECT numero_processo,
         data_decretao_falencia,
         dias_ate_convolucao,
         data_processamento_efetivo
  FROM default.gold_base_analitica
  WHERE classe_codigo = 129 AND escopo_lei_atual
)
SELECT
  (SELECT count(*) FROM base)                                              AS total_rj,
  (SELECT count(*) FROM base WHERE data_processamento_efetivo IS NOT NULL) AS com_concessao,
  count(CASE WHEN data_decretao_falencia IS NOT NULL THEN 1 END)           AS convoladas,
  round(count(CASE WHEN data_decretao_falencia IS NOT NULL THEN 1 END) * 100.0
        / (SELECT count(*) FROM base), 2)                                  AS taxa_bruta_pct,
  round(count(CASE WHEN data_decretao_falencia IS NOT NULL
                     AND data_processamento_efetivo IS NOT NULL THEN 1 END) * 100.0
        / NULLIF((SELECT count(*) FROM base
                  WHERE data_processamento_efetivo IS NOT NULL), 0), 2)    AS taxa_condicionada_pct,
  round(percentile_approx(CASE WHEN data_decretao_falencia IS NOT NULL
                              THEN dias_ate_convolucao END, 0.5), 1)       AS mediana_dias_ate_convolacao,
  round(avg(CASE WHEN data_decretao_falencia IS NOT NULL
                 THEN dias_ate_convolucao END), 1)                         AS media_dias_ate_convolacao,
  round(count(CASE WHEN data_decretao_falencia IS NOT NULL
                     AND data_processamento_efetivo IS NULL THEN 1 END) * 100.0
        / NULLIF(count(CASE WHEN data_decretao_falencia IS NOT NULL THEN 1 END), 0), 1)
                                                                           AS pct_convolacao_antes_da_concessao
FROM base;

total_rj,com_concessao,convoladas,taxa_bruta_pct,taxa_condicionada_pct,mediana_dias_ate_convolacao,media_dias_ate_convolacao,pct_convolacao_antes_da_concessao
5709,2422,177,3.10,5.12,1623,1980.2,29.9


#### Análise da resposta (pergunta 5)

**Resultado nacional (classe 129, Lei 11.101/2005):**
| Indicador | Valor |
|---|---|
| Total de RJs | 5.709 |
| RJs com concessão identificável | 2.422 |
| RJs convoladas em falência (mesmos autos, TPU 202) | 177 |
| **Taxa bruta (sobre todas as RJs)** | **3,10%** |
| **Taxa condicionada (sobre as com concessão)** | **5,12%** |
| Mediana do tempo até a convolação | 1.623 dias (~4,4 anos) |
| Média do tempo até a convolação | 1.980 dias (~5,4 anos) |
| Convolações sem concessão registrada | 29,9% (53 de 177) |

**Leitura:** a taxa condicionada (5,12%, medida só sobre as RJs com
concessão identificável) é a métrica recomendada — a convolação por
descumprimento pressupõe que o processo chegou à fase de processamento.
A sanidade interna confere: 177 = 124 com concessão + 53 sem.

**Observação:** 14 das 124 convoladas com concessão não têm a duração do stay computada na gold; elas permanecem na taxa condicionada, que conta eventos (concessão e decretação), não intervalos — o teste de coerência do intervalo só é aplicável na pergunta 2.


**Leitura das causas (aproximação temporal):** 29,9% das convolações ocorrem sem concessão registrada — candidatas a rejeição do plano/indeferimento (art. 73, V); os 70,1% restantes ocorrem após a concessão, perfil de descumprimento tardio durante a execução do plano. O tempo até a decretação (mediana de 1.623 dias) é medido sobre o conjunto completo das 177 convolações, incluindo as 53 sem concessão. Essa separação é inferência temporal, não medida direta.

**Coerência com as perguntas anteriores:** as 177 convoladas formam o universo onde o stay period se torna avaliável (pergunta 2) — medido de fato nas 110 com concessão e duração computadas, como lá se declara; e o total conversa com as 174 decretações que a decomposição da pergunta 4 mostrou entre os desfechos (diferença de 3 = decretadas ainda sem desfecho do processo registrado)



#### Resposta final (pergunta 5)

Entre 3,10% e 5,12% das Recuperações Judiciais do corpus terminam em
falência nos mesmos autos — a **taxa condicionada (5,12%), medida só
sobre as RJs com concessão identificável, é a métrica recomendada**. A
RJ retém a grande maioria das empresas que entram no sistema. O padrão
Datajud não registra a causa da convolação, mas a decomposição temporal
aproxima o diagnóstico: 29,9% das convolações ocorrem sem concessão
registrada (candidatas a rejeição/indeferimento, art. 73, V) e 70,1%
vêm anos após a concessão — perfil de descumprimento tardio do plano
durante a execução (o tempo até a decretação, mediana de 1.623 dias, é
medido sobre o conjunto completo das 177 convolações, incluindo as 53
sem concessão). 

Ressalvas: a causa é inferência temporal, não medida;
convolações em autos de falência separados ficam fora da taxa (piso
estrutural); o denominador condicionado herda a cobertura da concessão
da pergunta 1 (42,4%).

##### Nota comparativa com a pergunta 4
Sobre o tempo: a mediana de 1.623 dias daqui mede o intervalo ajuizamento → decretação da falência (dias_ate_convolucao); a pergunta 4 reporta 1.676 dias para a mesma rota porque lá a régua é dias_duracao_processo_total, que corre até o desfecho final do processo — podendo ser posterior à decretação (ex.: arquivamento). A diferença de ~53 dias é esperada, não é incoerência. A equivalência dos universos foi verificada: as 177 convoladas desta consulta são exatamente os mesmos processos com desfecho registrado na decomposição da pergunta 4.